# Challenges: can we believe the score?

```{admonition} Question
:class: tip
Can we believe the score?
```

**In this notebook:** 1 · two ways to split  2 · four models, tested twice  3 · other layouts  4 · three cities  5 · who mapped what  6 · a research case

## 1 · Two ways to split

A test hexagon picked at random almost always touches training hexagons, and neighbours are alike (data page).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict

import helpers as h

hexes, X, y, W = h.load()        # the table, the features, the labels, the neighbours
SEED = 0
xy = hexes[["x", "y"]].to_numpy()                # the centre of every hexagon, in metres

A random split deals single hexagons into five folds. **Spatial cross-validation** cuts the region into 10 blocks and deals whole blocks. Every fold is the test set once.

In [ ]:
N_BLOCKS = 10
blocks = KMeans(n_clusters=N_BLOCKS, n_init=10, random_state=SEED).fit_predict(xy)     # compact spatial blocks
random_folds = StratifiedKFold(5, shuffle=True, random_state=SEED)
spatial_folds = GroupKFold(5)

random_test = np.zeros(len(y), dtype=int)                 # for every hexagon: the fold in which it is tested
for k, (_, te) in enumerate(random_folds.split(X, y)):
    random_test[te] = k
spatial_test = np.zeros(len(y), dtype=int)
for k, (_, te) in enumerate(spatial_folds.split(X, y, groups=blocks)):
    spatial_test[te] = k
h.plot_folds(hexes, random_test, spatial_test);

<img src="_static/figures/4_folds_cycle.gif" alt="Random and spatial folds, one after the other" class="wide">

*The five test folds of each split in turn.*

## 2 · Four models, tested twice

A random forest gets four descriptions of a hexagon. The last holds only the coordinates. Every hexagon is predicted by a forest trained on the other folds.

In [ ]:
def cv_predict(F, folds, groups=None, trees=200):
    """One prediction per hexagon, made by a forest that was trained on the other folds."""
    return cross_val_predict(h.new_forest(trees), F, y, cv=folds, groups=groups)


feature_sets = {"POIs only": X, "POIs + neighbours' profile": np.hstack([X, W @ X]),
                "POIs + coordinates": np.hstack([X, xy]), "coordinates only": xy}
pred_random = {name: cv_predict(F, random_folds) for name, F in feature_sets.items()}
pred_spatial = {name: cv_predict(F, spatial_folds, blocks) for name, F in feature_sets.items()}
cv = pd.DataFrame({"random split": {n: (p == y).mean() for n, p in pred_random.items()},
                   "spatial split": {n: (p == y).mean() for n, p in pred_spatial.items()}})
cv["overstated by (points)"] = (cv["random split"] - cv["spatial split"]) * 100
h.plot_two_splits(cv, baseline=(y == "green").mean())
cv.round(3)

**Result.** POIs only: a random split adds 2 points. Coordinates only: 75% against 63%, 12 points. That model has memorised the map, and in an area it never saw it falls towards guessing.

## 3 · Other layouts

One layout of the blocks is one test. Five seeds and five layouts, with 100 trees:

In [ ]:
rows = []
for name in ["POIs only", "POIs + neighbours' profile", "coordinates only"]:
    for s in range(5):
        r = (cv_predict(feature_sets[name], StratifiedKFold(5, shuffle=True, random_state=s), trees=100) == y).mean()
        b = KMeans(n_clusters=N_BLOCKS, n_init=3, random_state=s).fit_predict(xy)
        sp = (cv_predict(feature_sets[name], GroupKFold(5), b, trees=100) == y).mean()
        rows += [{"features": name, "split": "random split", "accuracy": r}, {"features": name, "split": "spatial split", "accuracy": sp}]
runs = pd.DataFrame(rows)
summary = runs.groupby(["features", "split"], sort=False)["accuracy"].agg(["mean", "std"]).unstack()
h.plot_layouts(runs, summary)
(summary * 100).round(1)

**Result.** The order holds: 1, 2 and 16 points. A random split asks how well a model fills gaps in a sampled area, a spatial split how well it works somewhere new.

## 4 · Three cities

Train in two municipalities, predict the third. For comparison, a forest trained inside each city alone. **Balanced accuracy** averages, over the three classes, the share of each class that is found.

In [ ]:
city = hexes["municipality"].replace({"Kauniainen": "Espoo"}).to_numpy()      # Kauniainen (two hexagons) counts with Espoo
rows = []
for target in ["Helsinki", "Espoo", "Vantaa"]:
    te = city == target
    elsewhere = h.new_forest().fit(X[~te], y[~te]).predict(X[te])
    at_home = cross_val_predict(h.new_forest(), X[te], y[te], cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    rows.append({"tested on": target, "hexagons": int(te.sum()),
                 "largest class there": f"{pd.Series(y[te]).value_counts(normalize=True).iloc[0]:.0%}",
                 "trained in the other two": (elsewhere == y[te]).mean(),
                 "trained in this city only": (at_home == y[te]).mean(),
                 "balanced accuracy, trained elsewhere": balanced_accuracy_score(y[te], elsewhere)})
pd.DataFrame(rows).round(3)

**Result.** 73%, 83% and 75%, and a forest trained at home scores about the same: the land-use mix differs, the model travels. Balanced accuracy: 60% to 62%. A city unlike these three is not tested (**spatial heterogeneity**, week 3).

## 5 · Who mapped what

The features are what volunteers have mapped. The median residential hexagon of each city:

In [ ]:
detail = (hexes.assign(city=city)[hexes["landuse"] == "residential"]
          .groupby("city")[["poi_total", "street_furniture", "bike", "car", "shop_daily"]].median().astype(int))
share = hexes[["street_furniture", "car", "bike"]].sum().sum() / hexes[h.FEATURES].sum().sum()
print(f"Street furniture, car facilities and bicycle facilities make up {share:.0%} of all points of interest.")
print("Median number of POIs in a residential hexagon:")
detail

**Result.** Helsinki holds 26 pieces of street furniture and 7 bicycle facilities, Espoo and Vantaa 6 or 7 and 1. Denser or mapped in more detail: these data cannot tell. A model learns partly where mappers were busy (**representational bias**).

## 6 · A research case

From the author's research on transport emissions: predict the daily CO₂ of car trips that start in a hexagon (172 hexagons of about 5 km²) from its residents and jobs, with and without an embedding of travel times. R² on held-out hexagons:

| Features | Random split | Spatial split |
|---|---|---|
| residents and jobs | 0.67 | 0.63 |
| residents and jobs + travel-time embedding | 0.71 | 0.64 |

**Result.** The embedding adds 4 points under a random split and 1 under a spatial split. The gain appears only when test hexagons lie next to training hexagons.

*Source: emission accounting with the GEMMAT model (Dey, Marín-Flores and Tenkanen, 2026). Residents and jobs: Statistics Finland (Paavo 2024), building grid of HSY (2024).*

**Next:** [your own data](own_data).